# Verificacion Manual Menendez vs MT5

Notebook para comprobar manualmente en MT5 todos los valores relevantes de una barra candidata o una senal Menendez.

Flujo recomendado:
1. Elegir `symbol` y, si quieres, `timestamp_to_check`.
2. Revisar las tablas de `Ultimas senales` y `Ultimos setups`.
3. Comparar en MT5 la barra M30 seleccionada y la vela H4 mostrada en `H4_SOURCE_TIME`.
4. Validar que coinciden PSAR, SMAs, MACD, Stoch, Bollinger, pivots y estados estructurales.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)
pd.set_option('display.float_format', lambda value: f'{value:,.6f}')

cwd = Path.cwd().resolve()
root = next((path for path in [cwd, *cwd.parents] if (path / 'backtests' / 'menendez' / 'menendez_pipeline.py').exists()), None)
if root is None:
    raise RuntimeError('No se encontro la raiz del proyecto con menendez_pipeline.py')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from backtests.menendez.menendez_loader import cargar_bundle_menendez_symbol


In [ ]:
symbol = 'EURUSD.r'
timestamp_to_check = None  # Ejemplo: '2025-03-18 14:30:00'
bars_before = 20
bars_after = 10


In [ ]:
bundle = cargar_bundle_menendez_symbol(symbol)
df = bundle['processed'].copy()
df_htf = bundle['htf_context'].copy()

signal_cols = [
    'H4_SOURCE_TIME', 'H4_ATTRACTOR_DIR', 'H4_MACD_NEUTRAL', 'H4_STANDBY',
    'RETRACE_RATIO', 'FAN_BREAKOUT', 'MACD_TRIGGER', 'STOCH_TRIGGER',
    'RR_RATIO', 'SL_PRICE', 'TP_PRICE', 'TP_SOURCE', 'ENTRY_DIR', 'SETUP_ID'
]
setup_cols = [
    'H4_SOURCE_TIME', 'H4_ATTRACTOR_DIR', 'SETUP_ID', 'SETUP_DIR', 'SETUP_AGE',
    'W_ID', 'X_ID', 'RETRACE_RATIO', 'FAN_BREAKOUT', 'MACD_TRIGGER', 'STOCH_TRIGGER',
    'BASE_CHANNEL_STATE', 'DECEL_CHANNEL_STATE', 'RR_RATIO', 'ENTRY_READY'
]

signals = df.loc[df['ENTRY_READY'], [col for col in signal_cols if col in df.columns]].tail(20)
setups = df.loc[df['SETUP_ID'].notna(), [col for col in setup_cols if col in df.columns]].tail(30)

display(Markdown('## Ultimas senales'))
display(signals)
display(Markdown('## Ultimos setups evaluados'))
display(setups)


In [ ]:
if timestamp_to_check is None:
    if not signals.empty:
        selected_time = signals.index[-1]
    elif not setups.empty:
        selected_time = setups.index[-1]
    else:
        selected_time = df.index[-1]
else:
    selected_pos = df.index.get_indexer([pd.Timestamp(timestamp_to_check)], method='nearest')[0]
    selected_time = df.index[selected_pos]

selected_pos = df.index.get_loc(selected_time)
selected_row = df.loc[selected_time]
summary_cols = [
    'H4_SOURCE_TIME', 'H4_ATTRACTOR_DIR', 'H4_MACD_NEUTRAL', 'H4_STANDBY',
    'ENTRY_READY', 'ENTRY_DIR', 'SETUP_ID', 'SETUP_DIR', 'SETUP_AGE',
    'W_ID', 'X_ID', 'RETRACE_RATIO', 'FAN_BREAKOUT', 'MACD_TRIGGER', 'STOCH_TRIGGER',
    'BASE_CHANNEL_STATE', 'DECEL_CHANNEL_STATE', 'FRACTAL_SEGMENT_COUNT', 'FRACTAL_EQUIVALENT_CLASS',
    'RR_RATIO', 'SL_PRICE', 'TP_PRICE', 'TP_SOURCE'
]
summary_cols = [col for col in summary_cols if col in df.columns]

display(Markdown(f'## Barra seleccionada: `{selected_time}`'))
display(selected_row[summary_cols].to_frame(name='value'))


In [ ]:
start_pos = max(0, selected_pos - int(bars_before))
end_pos = min(len(df), selected_pos + int(bars_after) + 1)
window = df.iloc[start_pos:end_pos].copy()
window['SELECTED_BAR'] = window.index == selected_time

def show_group(title, columns, frame=window):
    cols = [col for col in columns if col in frame.columns]
    display(Markdown(f'## {title}'))
    display(frame[cols])

price_cols = [
    'open', 'high', 'low', 'close', 'spread', 'spread_price', 'SELECTED_BAR'
]
m30_indicator_cols = [
    'PSAR', 'PSAR_POLARITY', 'PSAR_FLIP_LONG', 'PSAR_FLIP_SHORT',
    'SMA_5', 'SMA_8', 'SMA_13', 'SMA_21', 'SMA_21_SLOPE', 'BULLISH_FAN', 'BEARISH_FAN',
    'MACD_LINE', 'MACD_SIGNAL', 'MACD_HIST',
    'STOCH_K', 'STOCH_D', 'STOCH_CROSS_UP', 'STOCH_CROSS_DOWN',
    'BB_MID', 'BB_UPPER', 'BB_LOWER',
    'D_PIVOT', 'D_R1', 'D_R2', 'D_S1', 'D_S2',
    'W_PIVOT', 'W_R1', 'W_R2', 'W_S1', 'W_S2'
]
structure_cols = [
    'H4_SOURCE_TIME', 'H4_ATTRACTOR_DIR', 'H4_MACD_NEUTRAL', 'H4_STANDBY',
    'M30_PSAR_POLARITY', 'W_ID', 'X_ID', 'W_START', 'W_END', 'X_END',
    'W_START_PRICE', 'W_END_PRICE', 'X_EXTREME_PRICE',
    'RETRACE_RATIO', 'FAN_BREAKOUT', 'BASE_CHANNEL_STATE', 'DECEL_CHANNEL_STATE',
    'MACD_TRIGGER', 'STOCH_TRIGGER', 'FRACTAL_SEGMENT_COUNT', 'FRACTAL_EQUIVALENT_CLASS',
    'SETUP_ID', 'SETUP_DIR', 'SETUP_AGE', 'ENTRY_READY', 'ENTRY_DIR'
]
risk_cols = [
    'SL_PRICE', 'TP_PRICE', 'TP_SOURCE', 'RR_RATIO',
    'TARGET_0.854', 'TARGET_1.0', 'TARGET_1.236', 'TARGET_1.618'
]
h4_sync_cols = [
    'H4_SOURCE_TIME', 'H4_CLOSE', 'H4_PSAR', 'H4_PSAR_POLARITY',
    'H4_SMA_5', 'H4_SMA_8', 'H4_SMA_13', 'H4_SMA_21', 'H4_SMA_21_SLOPE',
    'H4_BULLISH_FAN', 'H4_BEARISH_FAN',
    'H4_MACD_LINE', 'H4_MACD_SIGNAL', 'H4_MACD_HIST',
    'H4_MACD_NEUTRAL', 'H4_STANDBY', 'H4_ATTRACTOR_DIR'
]

show_group('Precio y spread M30', price_cols)
show_group('Indicadores M30', m30_indicator_cols)
show_group('Estructura y logica de setup', structure_cols)
show_group('Contexto H4 sincronizado usado por la decision', h4_sync_cols)
show_group('Gestion de riesgo y targets', risk_cols)


In [ ]:
source_time = selected_row.get('H4_SOURCE_TIME')
display(Markdown('## Vela H4 exacta usada por el sistema'))
if pd.notna(source_time) and pd.Timestamp(source_time) in df_htf.index:
    h4_row_cols = [
        'open', 'high', 'low', 'close', 'spread_price',
        'PSAR', 'PSAR_POLARITY',
        'SMA_5', 'SMA_8', 'SMA_13', 'SMA_21', 'SMA_21_SLOPE',
        'BULLISH_FAN', 'BEARISH_FAN',
        'MACD_LINE', 'MACD_SIGNAL', 'MACD_HIST', 'MACD_NEUTRAL', 'MACD_NEUTRAL_RUN', 'STANDBY',
        'ATTRACTOR_DIR'
    ]
    h4_row_cols = [col for col in h4_row_cols if col in df_htf.columns]
    display(df_htf.loc[[pd.Timestamp(source_time)], h4_row_cols].T.rename(columns={pd.Timestamp(source_time): 'value'}))
else:
    print('No hay vela H4 fuente disponible para esta barra.')
